# FAMPNN Tool Example

This notebook demonstrates how to use **FAMPNN** (Full-Atom MPNN) for protein sequence design with full-atom sidechain co-generation.

## What is FAMPNN?

[FAMPNN](https://github.com/richardshuai/fampnn) jointly models discrete amino acid identity and continuous sidechain conformation using combined cross-entropy and diffusion loss. Unlike backbone-only inverse folding models, FAMPNN generates both sequences and sidechain coordinates simultaneously.

### Key Features:
- **Full-atom design** -- Jointly generates sequences and sidechain coordinates
- **Sidechain packing** -- Predicts sidechain conformations for fixed sequences
- **Mutation scoring** -- Evaluates mutation fitness with full-atom context
- **Predicted sidechain error (pSCE)** -- Per-residue confidence scores for sidechain placement

### Operations:
| Tool Key | Operation | Description |
|----------|-----------|-------------|
| `fampnn-sample` | Sequence sampling | Design sequences with sidechain co-generation |
| `fampnn-pack` | Sidechain packing | Pack sidechains onto a fixed backbone + sequence |
| `fampnn-score` | Mutation scoring | Score specific mutations (log-likelihood ratios) |
| `fampnn-score-all-mutations` | Exhaustive scoring | Score all possible single mutations |

## Imports

In [1]:
from pathlib import Path

# Resolve path to example PDB relative to the bio_programming_tools package
import bio_programming_tools
from bio_programming_tools.entities.structures.structure import Structure
from bio_programming_tools.tools.inverse_folding.fampnn import (
    # Packing
    FAMPNNPackConfig,
    FAMPNNPackInput,
    FAMPNNPackStructureInput,
    # Sampling
    FAMPNNSampleConfig,
    FAMPNNSampleInput,
    # Score all mutations
    FAMPNNScoreAllMutationsConfig,
    FAMPNNScoreAllMutationsInput,
    # Scoring
    FAMPNNScoreConfig,
    FAMPNNScoreInput,
    FAMPNNStructureInput,
    MutationInput,
    run_fampnn_pack,
    run_fampnn_sample,
    run_fampnn_score,
    run_fampnn_score_all_mutations,
)

_REPO_ROOT = Path(bio_programming_tools.__file__).parent.parent
EXAMPLE_PDB = _REPO_ROOT / "tests" / "dummy_data" / "test_structure_similarity.pdb"
assert EXAMPLE_PDB.exists(), f"PDB not found: {EXAMPLE_PDB}"

## API Reference

### `FAMPNNStructureInput` (extends `InverseFoldingStructureInput`)

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structure` | `Structure` | *required* | Protein structure (file path, PDB string, or `Structure` object) |
| `chain_ids` | `Optional[List[str]]` | `None` | Chain IDs to design. If `None`, all chains are designed |
| `fixed_positions` | `Optional[Dict[str, List[int]]]` | `None` | Chain ID to residue positions (1-indexed) to keep fixed |
| `fixed_sidechain_positions` | `Optional[Dict[str, List[int]]]` | `None` | Residue positions with known sidechain coordinates to condition on |

### `FAMPNNSampleConfig` (extends `InverseFoldingConfig`)

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `num_sequences_per_structure` | `int` | `1` | Total sequences to generate per structure |
| `batch_size` | `Optional[int]` | `None` | Max sequences per GPU forward pass |
| `temperature` | `float` | `0.1` | Controls randomness in sampling (0.0 to 1.0) |
| `model_variant` | `str` | `"0.3"` | Model checkpoint: `"0.3"` (design), `"0.0"` (packing), `"0.3_cath"` (scoring) |
| `num_steps` | `int` | `100` | Number of iterative unmasking steps |
| `seq_only` | `bool` | `False` | Skip sidechain generation |
| `repack_last` | `bool` | `True` | Repack sidechains after final sequence |
| `psce_threshold` | `float` | `0.3` | Only keep sidechains below this error threshold |

### `FAMPNNSequences` (extends `DesignedSequences`)

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Designed amino acid sequences |
| `output_pdb_strings` | `List[str]` | PDB strings with designed sequences and sidechain coordinates |
| `psce` | `List[List[float]]` | Per-residue predicted sidechain error (Angstroms) |

## 1. Sequence Sampling

Design new protein sequences with full-atom sidechain co-generation. FAMPNN iteratively unmasks sequence positions while simultaneously denoising sidechain coordinates.

In [2]:
# Load a protein structure
structure = Structure(structure_filepath_or_content=EXAMPLE_PDB)
print(f"Loaded structure: chain(s) {structure.get_chain_ids()}, "
      f"{len(structure.get_chain_positions('A'))} residues")

# Set up input — design all chains
sample_input = FAMPNNSampleInput(
    inputs=[FAMPNNStructureInput(structure=structure)]
)

# Configure sampling
sample_config = FAMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 designs
    temperature=0.1,                # Conservative designs
    num_steps=10,                   # Unmasking steps (use 100 for production)
    seed=42,
)

# Run FAMPNN sampling
result = run_fampnn_sample(sample_input, sample_config)

# Inspect results
for i, designed in enumerate(result.designed_sequences):
    print(f"\nStructure {i} designs:")
    for j, seq in enumerate(designed.sequences):
        print(f"  Design {j+1}: {seq[:60]}..." if len(seq) > 60 else f"  Design {j+1}: {seq}")
        print(f"            Length: {len(seq)} residues")
        print(f"            Mean pSCE: {sum(designed.psce[j])/len(designed.psce[j]):.3f} A")

Loaded structure: chain(s) ['A'], 65 residues



Structure 0 designs:
  Design 1: MLRPLDLRAYVERHGITATARDLGLSRATVRRILARGEAVYVEVLEDGTRVTYVERDGRL...
            Length: 65 residues
            Mean pSCE: 1.550 A
  Design 2: MKKKLDIEAYVKKYGITKTAKELGLSKTTVRRIIKEKLPVYVETLEDGTKLTYVERNGKM...
            Length: 65 residues
            Mean pSCE: 1.514 A


## 2. Sampling with Fixed Positions

Fix specific residue positions during design to preserve important functional sites.

In [3]:
# Fix active site residues (1-indexed positions)
constrained_input = FAMPNNSampleInput(
    inputs=[FAMPNNStructureInput(
        structure=structure,
        chain_ids=["A"],
        fixed_positions={"A": [10, 15, 20, 25]},  # Keep these positions fixed
        fixed_sidechain_positions={"A": [10, 15]}, # Also fix sidechain conformations
    )]
)

constrained_config = FAMPNNSampleConfig(
    num_sequences_per_structure=1,
    temperature=0.1,
    num_steps=10,  # Use 100 for production
    seed=42,
)

result_fixed = run_fampnn_sample(constrained_input, constrained_config)
print(f"Designed {len(result_fixed.designed_sequences[0].sequences)} sequences with fixed positions")

Designed 1 sequences with fixed positions


## 3. Sidechain Packing

Given a backbone and sequence, predict sidechain coordinates using per-token Euclidean diffusion. The B-factor column in the output PDB contains per-atom pSCE.

In [4]:
# Pack sidechains for a structure
pack_input = FAMPNNPackInput(
    inputs=[FAMPNNPackStructureInput(structure=structure)]
)

pack_config = FAMPNNPackConfig(
    num_samples_per_structure=3,  # Generate 3 packing samples
    batch_size=3,
    seed=42,
)

pack_result = run_fampnn_pack(pack_input, pack_config)

# Inspect packing results
for i, (pdbs, psce_list) in enumerate(zip(pack_result.packed_structures, pack_result.psce)):
    print(f"Structure {i}: {len(pdbs)} packing samples")
    for j, psce in enumerate(psce_list):
        mean_psce = sum(psce) / len(psce)
        print(f"  Sample {j+1}: mean pSCE = {mean_psce:.3f} A")

# Save best packing to file
# with open("packed.pdb", "w") as f:
#     f.write(pack_result.packed_structures[0][0])

Structure 0: 3 packing samples
  Sample 1: mean pSCE = 1.428 A
  Sample 2: mean pSCE = 1.421 A
  Sample 3: mean pSCE = 1.420 A


## 4. Mutation Scoring

Score specific mutations using FAMPNN's full-atom context. Mutations use the format `<WT><0-indexed_pos><MUT>`. Scores are log-likelihood ratios (positive = favored over wild-type). Use `"wt"` to score the wild-type as a reference.

In [5]:
# Score specific mutations (0-indexed positions)
# The test structure starts with MRKKLD..., so M0V = Met->Val at pos 0, K2A = Lys->Ala at pos 2
score_input = FAMPNNScoreInput(
    inputs=[MutationInput(
        structure=structure,
        mutations=["M0V", "K2A", "wt"],  # 0-indexed positions
    )]
)

score_config = FAMPNNScoreConfig(seed=42)
score_result = run_fampnn_score(score_input, score_config)

# Display scores
for mut, score in zip(score_result.results[0].mutations, score_result.results[0].scores):
    label = "(wild-type)" if mut == "wt" else ""
    print(f"  {mut}: {score:+.4f} {label}")

  M0V: -1.3382 
  K2A: -3.0358 
  wt: +0.0000 (wild-type)


## 5. Score All Mutations

Generate a comprehensive mutational landscape by scoring every possible single amino acid substitution at every position.

In [6]:
# Score all possible single mutations
all_mut_input = FAMPNNScoreAllMutationsInput(inputs=[structure])
all_mut_config = FAMPNNScoreAllMutationsConfig(seed=42)

all_mut_result = run_fampnn_score_all_mutations(all_mut_input, all_mut_config)

# Display top beneficial mutations
scores = all_mut_result.results[0].scores
print(f"Scored {len(scores)} positions x 20 amino acids\n")

# Find most favorable mutations
all_scores = []
for pos_label, pos_scores in scores.items():
    wt_residue = pos_label[-1]  # Last char is wild-type
    for mut_aa, score in pos_scores.items():
        if mut_aa != wt_residue:
            all_scores.append((pos_label, mut_aa, score))

all_scores.sort(key=lambda x: x[2], reverse=True)
print("Top 5 beneficial mutations:")
for pos, aa, s in all_scores[:5]:
    print(f"  {pos} -> {aa}: {s:+.4f}")

Scored 65 positions x 20 amino acids

Top 5 beneficial mutations:
  15N -> G: +5.4041
  16Q -> I: +4.2020
  62P -> E: +3.8712
  13D -> E: +3.4600
  63F -> V: +3.4592


## 6. Export Results

Save results to files for downstream analysis.

In [7]:
# Export sampling results (FASTA format via InverseFoldingOutput)
result.export("fampnn_designs", export_path="./example_output", file_format="fasta")

# Export packing results (PDB files)
pack_result.export("fampnn_packed", export_path="./example_output", file_format="pdb")

# Export scoring results (CSV)
score_result.export("fampnn_scores", export_path="./example_output", file_format="csv")

# Export all-mutations results (CSV matrix)
all_mut_result.export("fampnn_all_mutations", export_path="./example_output", file_format="csv")

import os

print("Exported files:")
for root, dirs, files in os.walk("./example_output"):
    for f in sorted(files):
        filepath = os.path.join(root, f)
        size = os.path.getsize(filepath)
        print(f"  {filepath} ({size:,} bytes)")

Exported files:
  ./example_output/fampnn_packed/packed_0_sample_0.pdb (42,774 bytes)
  ./example_output/fampnn_packed/packed_0_sample_1.pdb (42,774 bytes)
  ./example_output/fampnn_packed/packed_0_sample_2.pdb (42,774 bytes)
  ./example_output/fampnn_scores/scores_0.csv (73 bytes)
  ./example_output/fampnn_designs/design_0.fasta (164 bytes)
  ./example_output/fampnn_all_mutations/all_scores_0.csv (24,541 bytes)
